# 01 – Data Profiling PaySim

Thực hiện: Khải; kiểm chứng/phụ trách hạng mục: Châm. Toàn bộ số liệu được tính từ CSV thật; không loại dữ liệu vì balance mismatch.

In [1]:
from pathlib import Path
import hashlib, json, math, re
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SEED = 42
CHUNK_SIZE = 200_000
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
CSV_PATH = next((RAW_DIR / n for n in ["PS_20174392719_1491204439457_log.csv", "PaySim.csv"] if (RAW_DIR / n).exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(f"Đặt PaySim tại {RAW_DIR} với một trong hai tên được hỗ trợ.")
OUT_DIR = ROOT / "docs" / "data"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_COLUMNS = ["step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","isFraud","isFlaggedFraud"]
header = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
assert header == EXPECTED_COLUMNS, f"Schema không đúng: {header}"
print(f"Nguồn: {CSV_PATH.name} | chunk={CHUNK_SIZE:,} | seed={SEED}")

Nguồn: PS_20174392719_1491204439457_log.csv | chunk=200,000 | seed=42


## 1. Profiling toàn bộ dữ liệu

Đọc tuần tự theo chunk; hash 64-bit ổn định được lưu cho toàn bộ 11 cột nên phát hiện được duplicate nằm ở các chunk khác nhau.

In [2]:
numeric_cols = ["step","amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest","isFraud","isFlaggedFraud"]
counts = Counter(); nulls = Counter(); domains = {c:set() for c in ["type","isFraud","isFlaggedFraud"]}
distinct_sets = {c:set() for c in ["type","nameOrig","nameDest","isFraud","isFlaggedFraud"]}
mins = {c:math.inf for c in numeric_cols}; maxs = {c:-math.inf for c in numeric_cols}; sums = Counter()
invalid = Counter(); hashes=[]; numeric_parts={c:[] for c in numeric_cols}
orig_prefix=Counter(); dest_prefix=Counter(); mismatch_orig=0; mismatch_dest=0; abnormal_balance=Counter()
for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
    counts["rows"] += len(chunk)
    for c in EXPECTED_COLUMNS: nulls[c] += int(chunk[c].isna().sum())
    for c in numeric_cols:
        a=chunk[c].to_numpy(); numeric_parts[c].append(a.copy()); mins[c]=min(mins[c],float(np.nanmin(a))); maxs[c]=max(maxs[c],float(np.nanmax(a))); sums[c]+=float(np.nansum(a))
    for c in distinct_sets: distinct_sets[c].update(chunk[c].dropna().unique().tolist())
    for c in domains: domains[c].update(chunk[c].dropna().unique().tolist())
    invalid["amount"] += int((~np.isfinite(chunk.amount) | (chunk.amount < 0)).sum())
    invalid["nameOrig"] += int((~chunk.nameOrig.str.match(r"^[CM]\d+$", na=False)).sum())
    invalid["nameDest"] += int((~chunk.nameDest.str.match(r"^[CM]\d+$", na=False)).sum())
    invalid["type"] += int((~chunk.type.isin(["CASH_IN","CASH_OUT","DEBIT","PAYMENT","TRANSFER"])).sum())
    invalid["isFraud"] += int((~chunk.isFraud.isin([0,1])).sum()); invalid["isFlaggedFraud"] += int((~chunk.isFlaggedFraud.isin([0,1])).sum())
    for k,v in chunk.nameOrig.str[0].value_counts().items(): orig_prefix[k]+=int(v)
    for k,v in chunk.nameDest.str[0].value_counts().items(): dest_prefix[k]+=int(v)
    abnormal_balance["negative"] += int((chunk[["oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]] < 0).any(axis=1).sum())
    mismatch_orig += int((~np.isclose(chunk.oldbalanceOrg - chunk.amount, chunk.newbalanceOrig, atol=.01)).sum())
    mismatch_dest += int((~np.isclose(chunk.oldbalanceDest + chunk.amount, chunk.newbalanceDest, atol=.01)).sum())
    hashes.append(pd.util.hash_pandas_object(chunk[EXPECTED_COLUMNS], index=False).to_numpy(dtype="uint64"))
all_hashes=np.concatenate(hashes); _, hash_counts=np.unique(all_hashes, return_counts=True)
duplicate_rows=int((hash_counts-1).clip(min=0).sum()); duplicate_groups=int((hash_counts>1).sum())
arrays={c:np.concatenate(v) for c,v in numeric_parts.items()}
print(f"Dòng={counts['rows']:,}; cột={len(EXPECTED_COLUMNS)}; duplicate dư={duplicate_rows:,}; nhóm duplicate={duplicate_groups:,}")

Dòng=6,362,620; cột=11; duplicate dư=0; nhóm duplicate=0


## 2. Bảng profiling và kiểm tra chất lượng

In [3]:
rows=[]
for c in EXPECTED_COLUMNS:
    is_num=c in numeric_cols; a=arrays[c] if is_num else None
    distinct=len(np.unique(a)) if is_num else len(distinct_sets[c])
    rows.append({"column":c,"dtype":str(pd.read_csv(CSV_PATH,nrows=1)[c].dtype),"non_null":counts["rows"]-nulls[c],"null_count":nulls[c],"null_pct":nulls[c]/counts["rows"]*100,"distinct":distinct,
      "min":float(np.nanmin(a)) if is_num else min(distinct_sets[c]),"max":float(np.nanmax(a)) if is_num else max(distinct_sets[c]),
      "mean":float(np.nanmean(a)) if is_num else "","median":float(np.nanmedian(a)) if is_num else "","p25":float(np.nanpercentile(a,25)) if is_num else "","p75":float(np.nanpercentile(a,75)) if is_num else "","p95":float(np.nanpercentile(a,95)) if is_num else "","p99":float(np.nanpercentile(a,99)) if is_num else "",
      "invalid_count":invalid.get(c,0),"note": {"step":"1 step = 1 giờ mô phỏng","type":f"Domain: {sorted(domains['type'])}","amount":f"Tổng amount: {sums['amount']:.2f}","nameOrig":f"Prefix: {dict(orig_prefix)}","nameDest":f"Prefix: {dict(dest_prefix)}","isFraud":f"Domain: {sorted(domains['isFraud'])}","isFlaggedFraud":f"Domain: {sorted(domains['isFlaggedFraud'])}"}.get(c,"")})
profile=pd.DataFrame(rows); profile.to_csv(OUT_DIR/"profiling_summary.csv",index=False,encoding="utf-8-sig")
display(profile)
quality={"source":CSV_PATH.name,"run_date":pd.Timestamp.now().date().isoformat(),"rows":counts["rows"],"columns":len(EXPECTED_COLUMNS),"total_amount":sums["amount"],"fraud":int(sums["isFraud"]),"flagged":int(sums["isFlaggedFraud"]),"duplicate_rows":duplicate_rows,"duplicate_groups":duplicate_groups,"negative_balance_rows":abnormal_balance["negative"],"origin_balance_mismatch":mismatch_orig,"destination_balance_mismatch":mismatch_dest,"orig_prefix":dict(orig_prefix),"dest_prefix":dict(dest_prefix),"domains":{k:sorted(v) for k,v in domains.items()}}
(OUT_DIR/"profiling_metrics.json").write_text(json.dumps(quality,ensure_ascii=False,indent=2),encoding="utf-8")
display(Markdown(f"**Balance mismatch không bị loại bỏ:** nguồn {mismatch_orig:,} dòng; đích {mismatch_dest:,} dòng. Đây có thể là đặc tính mô phỏng/ghi nhận số dư của PaySim, không đủ căn cứ coi là dữ liệu lỗi."))

,column,dtype,non_null,null_count,null_pct,distinct,min,max,mean,median,p25,p75,p95,p99,invalid_count,note
0,step,int64,6362620,0,0.0,743,1.0,743.0,243.397246,239.0,156.0,335.0,490.0,681.0,0,1 step = 1 giờ mô phỏng
1,type,str,6362620,0,0.0,5,CASH_IN,TRANSFER,,,,,,,0,"Domain: ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYM..."
2,amount,float64,6362620,0,0.0,5316900,0.0,92445516.64,179861.903549,74871.94,13389.57,208721.4775,518634.1965,1615979.4716,0,Tổng amount: 1144392944759.77
3,nameOrig,str,6362620,0,0.0,6353307,C1000000639,C999999784,,,,,,,0,Prefix: {'C': 6362620}
4,oldbalanceOrg,float64,6362620,0,0.0,1845844,0.0,59585040.37,833883.104074,14208.0,0.0,107315.175,5823702.2785,16027256.1337,0,
5,newbalanceOrig,float64,6362620,0,0.0,2682586,0.0,49585040.37,855113.668579,0.0,0.0,144258.41,5980262.3365,16176160.558,0,
6,nameDest,str,6362620,0,0.0,2722362,C1000004082,M999999784,,,,,,,0,"Prefix: {'C': 4211125, 'M': 2151495}"
7,oldbalanceDest,float64,6362620,0,0.0,3614697,0.0,356015889.35,1100701.66652,132705.665,0.0,943036.7075,5147229.7135,12371819.1548,0,
8,newbalanceDest,float64,6362620,0,0.0,3555499,0.0,356179278.92,1224996.398202,214661.44,0.0,1111909.25,5515715.9035,13137866.941,0,
9,isFraud,int64,6362620,0,0.0,2,0.0,1.0,0.001291,0.0,0.0,0.0,0.0,0.0,0,"Domain: [0, 1]"


**Balance mismatch không bị loại bỏ:** nguồn 5,000,754 dòng; đích 3,823,898 dòng. Đây có thể là đặc tính mô phỏng/ghi nhận số dư của PaySim, không đủ căn cứ coi là dữ liệu lỗi.

In [4]:
assert counts["rows"] == 6_362_620 and int(sums["isFraud"]) == 8_213
assert profile.shape[0] == 11 and profile.null_count.sum() == 0
print("PASS: dataset PaySim chuẩn; CSV profiling đã ghi và kiểm tra.")

PASS: dataset PaySim chuẩn; CSV profiling đã ghi và kiểm tra.
